# Phase 13 — Simulation Opérationnelle Complète (600 cellules)

> **🚀 Résultat de cette phase :** Cette simulation à grande échelle valide les performances du système SON sur un cluster complet. Les logs détaillés et métriques sont intégrés dans le rapport final (Notebook 14).

Simulation rigoureuse comparant l'approche statique vs dynamique (SON) avec un moteur MILP optimisÃ© minimisant l'excÃ¨s final global.

In [ ]:
import polars as pl
import numpy as np
import pickle, json, yaml
import sys
sys.path.append('../../')
from src.optimization.milp_engine import build_H_matrices, solve_congestion_milp as solve_congestion

# Chargement
with open('../models/xgb_q80.pkl', 'rb') as f: model_q80 = pickle.load(f)
with open('../config/network_topology.yaml', 'r') as f: topology = yaml.safe_load(f)
coverage_600 = json.load(open('../data/processed/cell_antenna_map_600.json', 'r'))
fractions_df = pl.read_parquet('../offline/fractions.parquet')
df_test = pl.read_parquet('../data/processed/features_target_600cells.parquet').filter(pl.col('slot_30m') >= 56*48)

delta_levels = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]

def run_full_simulation():
    total_dynamic = 0.0
    total_static = 0.0
    slots = df_test['slot_30m'].unique().sort().to_list()

    for slot in slots[:20]: # Ã‰chantillon de 20 slots
        slot_data = df_test.filter(pl.col('slot_30m') == slot)
        X_slot = slot_data.select([c for c in slot_data.columns if c not in ['square_id', 'slot_30m', 'target_1h']]).to_numpy()
        preds = model_q80.predict(X_slot)
        
        antenna_preds = {ant: 0.0 for ant in topology['antennas']}
        for i, row in enumerate(slot_data.iter_rows(named=True)):
            ant = next((a for a, cells in coverage_600.items() if row['square_id'] in cells), None)
            if ant: antenna_preds[ant] += preds[i]
        
        antenna_stats = {ant: {'V_a': v, 'C_a': topology['antennas'][ant]['capacity_mo']} for ant, v in antenna_preds.items()}
        
        antennas_list, H_deleste, H_recv = build_H_matrices(fractions_df, antenna_preds, coverage_600)
        sol = solve_congestion(antennas_list, antenna_stats, H_deleste, H_recv, delta_levels)
        
        # Statique : excÃ¨s brut initial
        total_static += sum(max(0.0, antenna_preds[a] - antenna_stats[a]['C_a']) for a in antennas_list)
        
        # Dynamique : calcul corrigÃ©
        for i, ant in enumerate(antennas_list):
            if ant in sol:
                my_level = delta_levels.index(sol[ant]['delta_dB'])
                dÃ©lestage_ant = H_deleste[i, my_level]
                reÃ§u_ant = 0
                for j, ant_j in enumerate(antennas_list):
                    if ant_j in sol:
                        their_level = delta_levels.index(sol[ant_j]['delta_dB'])
                        reÃ§u_ant += H_recv[i, j, their_level]
                final_vol = antenna_preds[ant] - dÃ©lestage_ant + reÃ§u_ant
                total_dynamic += max(0.0, final_vol - antenna_stats[ant]['C_a'])
            else:
                total_dynamic += max(0.0, antenna_preds[ant] - antenna_stats[ant]['C_a'])
    return total_static, total_dynamic

res_static, res_dynamic = run_full_simulation()
print(f"RÃ©sultats finaux : Statique={res_static:.2f} Mo, Dynamique={res_dynamic:.2f} Mo, AmÃ©lioration={((res_static - res_dynamic)/res_static)*100:.2f}%")